In [1]:
import pandas as pd
from scripts.prepare_data import prepare_raw_data
from scripts.feature_engineering import add_categories, add_target_lags
from scripts.evaluate import calculate_metrics

In [2]:
df_items = pd.read_csv('data/landing/items.csv')
df_cat = pd.read_csv('data/landing/item_categories.csv')

In [3]:
df = prepare_raw_data('data/landing/sales_train.csv')
df.head()

,date_block_num,shop_id,item_id,target
0,0,2,19,0.0
1,0,2,27,1.0
2,0,2,28,0.0
3,0,2,29,0.0
4,0,2,32,0.0


In [4]:
df = add_categories(df, df_cat, df_items)
df = add_target_lags(df, (1,2,3,6,12))
df.head()

,date_block_num,shop_id,item_id,target,item_category_id,category_type_id,item_type_id,target_lag_1,target_lag_2,target_lag_3,target_lag_6,target_lag_12
0,0,2,19,0.0,40,12,7,NaN,NaN,NaN,NaN,NaN
1,0,2,27,1.0,19,6,13,NaN,NaN,NaN,NaN,NaN
2,0,2,28,0.0,30,9,54,NaN,NaN,NaN,NaN,NaN
3,0,2,29,0.0,23,6,19,NaN,NaN,NaN,NaN,NaN
4,0,2,32,0.0,40,12,7,NaN,NaN,NaN,NaN,NaN


In [5]:
m = df['date_block_num']
train = df[(m >= 12) & (m <= 31)] 
valid = df[m == 32]                
test  = df[m == 33]                

In [6]:
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import root_mean_squared_error

CAT = ['shop_id', 'item_category_id', 'category_type_id', 'item_type_id']
FEATURES = CAT + ['target_lag_1', 'target_lag_3', 'target_lag_6', 'target_lag_12']

m = df['date_block_num']
train, valid = df[(m >= 12) & (m <= 31)], df[m == 32]

model = HistGradientBoostingRegressor(
    learning_rate=0.1, max_iter=300, max_leaf_nodes=63, min_samples_leaf=100,
    categorical_features=CAT,
    early_stopping=False,  # встроенный отбирает случайные 10% строк, это смешивает месяцы
    random_state=42, verbose=1,
)
model.fit(train[FEATURES], train['target'])

curve = [root_mean_squared_error(valid['target'], np.clip(p, 0, 20))
         for p in model.staged_predict(valid[FEATURES])]
best = int(np.argmin(curve)) + 1
print(f'best_iter={best}, rmse={curve[best - 1]:.4f}')

for name, p in {'zeros': np.zeros(len(valid)), 'naive': valid['target_lag_1'].fillna(0)}.items():
    print(name, round(root_mean_squared_error(valid['target'], p), 4))

Binning 0.381 GB of training data: 0.205 s
Fitting gradient boosted rounds:
Fit 300 trees in 40.907 s, (18900 total leaves)
Time spent computing histograms: 20.850s
Time spent finding best splits:  0.380s
Time spent applying splits:      7.884s
Time spent predicting:           3.304s
best_iter=207, rmse=0.8774
zeros 1.1982
naive 1.0766


In [7]:
result = calculate_metrics(
    model_name='HistGradientBoosting',
    model=model,
    backend='plotly',
    splits={
        'train': (train[FEATURES], train['target']),
        'valid': (valid[FEATURES], valid['target'])
    },
    baselines={
        'zeros' : np.zeros(len(valid)),
        'naive': valid['target_lag_1'].fillna(0) 
    }
)
result.round(4)

,rmse,mae,bias,r2,rmse_nonzero
train,0.8531,0.3206,-0.0001,0.4918,2.0675
valid,0.8776,0.3178,0.0137,0.4349,2.1728
valid: zeros,1.1982,0.2701,-0.2701,-0.0536,3.2177
valid: naive,1.0766,0.3202,0.0106,0.1494,2.6185
